In [3]:
from evolutionSimulation.python.neuralnetworks.nn import * 
from datasets import load_dataset
import torch
import transformers

brain = Brain()
data = load_dataset("json", data_files=r"C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\python\dataset\simple_dataset.json")
shuffled_dataset = data.shuffle()

In [20]:
import torch
import time
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
import os

animals = {
    "lion": 1,
    "crocodile": 1,
    "dragon": 1,
    "duck": 0,
    "sheep": 0,
}

def batch(batch_size, start_index, dataset):
    truth = []
    images = [sample for sample in dataset["train"][start_index:start_index + batch_size]["image"]]
    tensor = torch.tensor(images)
    tensor = tensor.view(10, 1, 28, 28)
    for animal in dataset["train"][start_index:start_index + batch_size]["name"]:
        truth.append(animals[animal])
    return tensor.float(), truth

def train(num_img, batch_size, num_epoch, model, dataset):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.Adam(model.parameters(), lr=1e-5)
    model.to(device)
    model.train()
    best_loss = float("inf")
    total_loss = 0

    for epoch in range(num_epoch):
        epoch_loss = 0
        t0 = time.perf_counter()

        for i in range(0, num_img, batch_size):
            temp_batch = batch(batch_size, i, dataset)
            predictions = model(temp_batch[0].to(device))
            ground_truth = torch.tensor(temp_batch[1]).to(device, dtype = torch.long)
            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(predictions, ground_truth)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            if (i / num_img * 100) % 10 == 0:
                print(f"{i / num_img * 100}% | Loss: {loss.item():.4f}")
        
        avg_loss = epoch_loss / (num_img // batch_size)
        total_loss += epoch_loss
        t1 = time.perf_counter()
        print(f"Finished Epoch {epoch} in {t1 - t0} seconds, Loss: {avg_loss:.4f}")
        try: 
            os.mkdir(r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\{}'.format(num_img))
        except FileExistsError:
            pass
        torch.save(model.state_dict(), r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\{}\model{}.pt'.format(num_img, epoch))


In [23]:
train(10000, 10, 3,  brain, shuffled_dataset)

0.0% | Loss: 0.2853
10.0% | Loss: 0.2894
20.0% | Loss: 0.3100
30.0% | Loss: 0.1398
40.0% | Loss: 0.5123
50.0% | Loss: 0.0140
60.0% | Loss: 0.0135
70.0% | Loss: 0.6171
80.0% | Loss: 0.0627
90.0% | Loss: 0.0268
Finished Epoch 0 in 10.004817799999728 seconds, Loss: 0.2175
0.0% | Loss: 0.2186
10.0% | Loss: 0.3012
20.0% | Loss: 0.2997
30.0% | Loss: 0.1306
40.0% | Loss: 0.4966
50.0% | Loss: 0.0137
60.0% | Loss: 0.0146
70.0% | Loss: 0.5888
80.0% | Loss: 0.0699
90.0% | Loss: 0.0266
Finished Epoch 1 in 10.547485099999903 seconds, Loss: 0.2108
0.0% | Loss: 0.2066
10.0% | Loss: 0.2836
20.0% | Loss: 0.2772
30.0% | Loss: 0.1218
40.0% | Loss: 0.4863
50.0% | Loss: 0.0135
60.0% | Loss: 0.0154
70.0% | Loss: 0.5608
80.0% | Loss: 0.0751
90.0% | Loss: 0.0266
Finished Epoch 2 in 20.907139400000233 seconds, Loss: 0.2049


In [19]:
brain = Brain()
brain.load_state_dict(torch.load(r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\10\model2.pt'))

<All keys matched successfully>

In [6]:
import numpy as np
test2 = np.array(shuffled_dataset["train"][200002]["image"], dtype=np.uint8 )
print(shuffled_dataset["train"][10004]["name"])

sheep


In [15]:
rand = np.random.randint(0, 10000)
print(shuffled_dataset["train"][rand]["name"])
print(brain(torch.tensor(shuffled_dataset["train"][rand]["image"]).view(1, 1, 28, 28).float()))

lion
tensor([[-3.4733,  0.5263]], grad_fn=<AddmmBackward0>)
